In [2]:
import io
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import requests

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 120

RUN_ID = 22193615519
FULL_NAME = "MetaMask/metamask-mobile"


# Save here (your requested folder)
OUT_BASE = Path(r"C:\Android Mobile App\ICST2026_Ext\log")
OUT_BASE.mkdir(parents=True, exist_ok=True)

OUT_ZIP = OUT_BASE / f"run_{RUN_ID}_logs.zip"
OUT_DIR = OUT_BASE / f"run_{RUN_ID}_logs_extracted"

# =========================
# Token loader (same as Stage code)
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# =========================
# Minimal GitHub client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "log-downloader/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                return i
        return 0

    def request_logs_zip(self, full_name: str, run_id: int) -> Tuple[Optional[bytes], str]:
        url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/logs"

        idx = self._pick_idx()
        st = self.tokens[idx]
        self.session.headers["Authorization"] = f"Bearer {st.token}"

        try:
            r = self.session.get(url, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S), allow_redirects=False)
        except requests.exceptions.RequestException:
            return None, "error_request"

        rem = r.headers.get("X-RateLimit-Remaining")
        if rem is not None:
            try:
                st.remaining = int(rem)
            except Exception:
                pass

        if r.status_code == 404:
            return None, "not_found"
        if r.status_code == 403:
            return None, "forbidden"

        # Usually 302 redirect to signed URL
        if r.status_code in (301, 302, 303, 307, 308):
            loc = r.headers.get("Location") or r.headers.get("location")
            if not loc:
                return None, f"error_{r.status_code}_no_location"
            try:
                r2 = requests.get(loc, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                return None, "error_signed_request"
            if r2.status_code == 200:
                return r2.content, "ok"
            if r2.status_code == 404:
                return None, "not_found"
            if r2.status_code == 403:
                return None, "forbidden"
            return None, f"error_signed_{r2.status_code}"

        if r.status_code == 200:
            return r.content, "ok"

        return None, f"error_{r.status_code}"

# =========================
# Main
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    data, status = gh.request_logs_zip(FULL_NAME, RUN_ID)
    print("download status:", status)
    if status != "ok" or not data:
        raise SystemExit("Failed to download logs zip.")

    OUT_ZIP.write_bytes(data)
    print("saved zip:", OUT_ZIP)

    if OUT_DIR.exists():
        # If you rerun, keep things clean
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                p.unlink()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(data), "r") as z:
        z.extractall(OUT_DIR)

    print("extracted to:", OUT_DIR)

if __name__ == "__main__":
    main()


download status: ok
saved zip: C:\Android Mobile App\ICST2026_Ext\log\run_22193615519_logs.zip
extracted to: C:\Android Mobile App\ICST2026_Ext\log\run_22193615519_logs_extracted
